In [1]:
import geopandas as gpd
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import os
import time

# 1. SETUP INPUTS
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", ".."))

outpath = os.path.join(REPO_ROOT, "data", "pillar2_data")
geojson_file_path = os.path.join(REPO_ROOT, "data", "misc", "adm0.geojson")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/xml, text/xml, */*",
    "Accept-Language": "en-US,en;q=0.9",
}

# 2. DEFINE VARIABLES AND URLS
# {iso3} replaced with "" = SDMX wildcard for all countries (1 request per indicator)
p2_variables = {
    "P2_At-least_basic_sanitation.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,WASH_HOUSEHOLDS,1.0/{iso3}.WS_PPL_S-ALB.._T._T",
    "P2_At-least_basic_drinking_water.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,WASH_HOUSEHOLDS,1.0/{iso3}.WS_PPL_W-ALB.WAT._T._T",
    "P2_Basic_hygiene.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,WASH_HOUSEHOLDS,1.0/{iso3}.WS_PPL_H-B.HYG._T._T",
    "P2_Under_five_mortality.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,CME,1.0/{iso3}.CME_MRY0T4._T._T",
    "P2_Under_five_covered_by_social_protection.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,SOC_PROTECTION,1.0/{iso3}.SPP_CHLD_SOC_PROT.._T._T._T",
    "P2_Child_poverty.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,CHLD_PVTY,1.0/{iso3}.PV_CHLD_DPRV-S-L1-HS._T._T",
    "P2_Child_marriage.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,PT_CM,1.0/{iso3}.PT_F_20-24_MRD_U18._T.Y20T24._T._T._T._T._T._T._T._T",
    "P2_Child_labor.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,PT,1.0/{iso3}.PT_CHLD_5-17_LBR_ECON-HC._T.Y5T17._T._T._T._T",
    "P2_Learning_poverty.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,EDUCATION_FLS,1.0/{iso3}.ED_SE_LPV_PRIM._T",
    "P2_Lower_secondary_completion_rate.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,EDUCATION,1.0/{iso3}.ED_CR_L2._T.ISCED11_2._T._T.PCNT",
    "P2_Lower_secondary_out_of_school.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,EDUCATION,1.0/{iso3}.ED_ROFST_L2._T.ISCED11_2._T._T.PCNT",
    "P2_Child_Food_Poverty.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,NUTRITION,1.0/{iso3}.NT_CF_FG_0_T_2._T._T._T._T._T._T",
    "P2_Stunting.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,NUTRITION,1.0/{iso3}.NT_ANT_HAZ_NE2_MOD._T._T._T._T._T._T",
    "P2_Skilled_birth_coverage.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,MNCH,1.0/{iso3}.MNCH_SAB._T.Y15T49._T._T._T._T.",
    "P2_DTP3_access.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,IMMUNISATION,1.0/{iso3}.IM_DTP3.DTP3.M12T23",
    "P2_DTP1_access.csv": "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,IMMUNISATION,1.0/{iso3}.IM_DTP1.DTP1.M12T23"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)


def fetch_all_countries(api_url_template, retries=3, backoff=15):
    """Fetch all countries in one SDMX request (empty first dimension = wildcard)."""
    api_url = api_url_template.format(iso3="")

    for attempt in range(retries):
        try:
            response = SESSION.get(api_url, params={"format": "sdmx-compact-2.1"}, timeout=120)

            if response.status_code == 404:
                print("  No data for this indicator (404).")
                return []

            if response.status_code in (403, 429, 503):
                wait = backoff * (2 ** attempt)
                print(f"  HTTP {response.status_code} (attempt {attempt+1}), retrying in {wait}s...")
                time.sleep(wait)
                continue

            if response.status_code != 200:
                print(f"  HTTP {response.status_code} — skipping.")
                return []

            try:
                root = ET.fromstring(response.content)
            except ET.ParseError as pe:
                print(f"  XML parse error: {pe}")
                return []

            country_latest = {}
            for series in root.findall(".//Series"):
                iso3 = series.get("REF_AREA")
                if not iso3:
                    continue
                for obs in series.findall("Obs"):
                    try:
                        time_period = int(obs.get("TIME_PERIOD"))
                        obs_value = float(obs.get("OBS_VALUE"))
                    except Exception:
                        continue
                    if iso3 not in country_latest or time_period > country_latest[iso3]["time_period"]:
                        country_latest[iso3] = {
                            "iso3": iso3,
                            "time_period": time_period,
                            "obs_value": obs_value,
                            "data_source": obs.get("DATA_SOURCE") or series.get("DATA_SOURCE"),
                        }

            return list(country_latest.values())

        except requests.exceptions.RequestException as e:
            print(f"  Request error (attempt {attempt+1}): {e}")
            time.sleep(backoff)

    print(f"  Gave up after {retries} attempts.")
    return []


# 3. LOAD GEODATA
try:
    countries_gdf = gpd.read_file(geojson_file_path)
    print("GeoDataFrame loaded.")
    iso3_codes = set(countries_gdf["ISO3"].dropna().unique())
    print(f"Found {len(iso3_codes)} countries in reference file.")
except Exception as e:
    print(f"Error loading GeoJSON: {e}")
    iso3_codes = set()

# 4. MAIN LOOP — one request per indicator
os.makedirs(outpath, exist_ok=True)
for filename, api_template in p2_variables.items():
    print(f"\n==========================================")
    print(f"Processing: {filename}")
    print(f"==========================================")

    all_data = fetch_all_countries(api_template)

    if all_data:
        df = pd.DataFrame(all_data)
        if iso3_codes:
            df = df[df["iso3"].isin(iso3_codes)]
        csv_file_path = os.path.join(outpath, filename)
        df.to_csv(csv_file_path, index=False)
        print(f"Exported {len(df)} records to {csv_file_path}")
    else:
        print(f"Warning: No data fetched for {filename}")


GeoDataFrame loaded.
Found 262 countries in reference file.

Processing: P2_At-least_basic_sanitation.csv
Exported 230 records to /mnt/pixel_aid_disk1/projects/CCRR/data/pillar2_data/P2_At-least_basic_sanitation.csv

Processing: P2_At-least_basic_drinking_water.csv
Exported 230 records to /mnt/pixel_aid_disk1/projects/CCRR/data/pillar2_data/P2_At-least_basic_drinking_water.csv

Processing: P2_Basic_hygiene.csv
Exported 120 records to /mnt/pixel_aid_disk1/projects/CCRR/data/pillar2_data/P2_Basic_hygiene.csv

Processing: P2_Under_five_mortality.csv
Exported 199 records to /mnt/pixel_aid_disk1/projects/CCRR/data/pillar2_data/P2_Under_five_mortality.csv

Processing: P2_Under_five_covered_by_social_protection.csv
Exported 182 records to /mnt/pixel_aid_disk1/projects/CCRR/data/pillar2_data/P2_Under_five_covered_by_social_protection.csv

Processing: P2_Child_poverty.csv
Exported 72 records to /mnt/pixel_aid_disk1/projects/CCRR/data/pillar2_data/P2_Child_poverty.csv

Processing: P2_Child_marri